# NB03 — The Oxygen Attack on Ethylene: Why Transition States Are Multireference

In this notebook we study the **reaction pathway of atomic oxygen** O(³P) attacking the C=C double bond of **ethylene**.

This is a real chemical example where three levels of theory give qualitatively different answers:

| Method | Barrier? | Why? |
|--------|----------|------|
| B3LYP | ❌ None | Single-reference, misses open-shell character |
| CASSCF(6,6) | ✅ Yes, but ~90 kJ/mol | Multireference, but missing dynamic correlation |
| CASSCF + NEVPT2 | ✅ ~50 kJ/mol | Multireference + dynamic correlation |

The notebook proceeds in five stages:
1. Coarse CASSCF relaxed scan (3.5 → 1.45 Å, 10 points)
2. NEVPT2 single points on coarse geometries
3. Dense CASSCF relaxed scan around the barrier (2.4 → 1.45 Å, 15 points)
4. NEVPT2 and B3LYP single points on all geometries + final plot
5. Interactive trajectory animation linked to the energy curve

**Note on B3LYP:** The SCF wavefunction is unstable (negative stability eigenvalue) at several
geometries in the approach region — ORCA aborts those jobs after 5 reconvergence attempts.
This is itself a telling diagnostic: B3LYP cannot even find a stable wavefunction where
the multireference character is strongest.


## 0. Imports and setup


In [ ]:
import sys
sys.path.insert(0, '../tools')

import shutil
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, SVG
from pathlib import Path

from utils import (
    ORCA, NPROCS, HARTREE_TO_KJMOL,
    setup_workdir, run_orca,
    get_energy, get_nevpt2_energy,
    get_distance, terminated_normally
)
from qctools import load_xyz_as_traj, build_xyz_trajectory

# O-C1 distance extractor — atoms 0 and 2 in the xyz files
get_oc_distance = lambda f: get_distance(f, 0, 2)

# Include %pal block only when more than one core is available
pal_block = f'%pal nprocs {NPROCS} end\n\n' if NPROCS > 1 else ''

print(f'ORCA:   {ORCA}')
print(f'NPROCS: {NPROCS}')

## 1. Working directory


In [ ]:
WORKDIR = 'ethylene'
FORCE_CLEAN = False  # set True to start from scratch

if FORCE_CLEAN and Path(WORKDIR).exists():
    shutil.rmtree(WORKDIR)
    print(f'Removed {WORKDIR}/')

work_dir = setup_workdir(WORKDIR)
print(f'Working in: {work_dir}')

## 2. The reaction

The reaction is O(³P) + ethylene → triplet biradical intermediate.
The overall system has **charge 0, multiplicity 3** throughout the scan
(triplet oxygen + singlet ethylene = triplet).

The **curly arrows** below show how electrons move:
- The **blue arrow** traces one pair of π electrons as they form the new O–C1 bond
- The **red arrow** traces the remaining π electron as it becomes an unpaired radical on C2

The result is a **triplet biradical** — one unpaired electron on O, one on C2.
This is the species whose formation we are computing along the reaction path.


In [ ]:
reaction_scheme = """
<svg width="100%" viewBox="0 0 680 380" xmlns="http://www.w3.org/2000/svg">
<defs>
  <marker id="arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
    <path d="M2 1L8 5L2 9" fill="none" stroke="context-stroke" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
  </marker>
  <marker id="curl" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse">
    <path d="M2 1L8 5L2 9" fill="none" stroke="context-stroke" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"/>
  </marker>
</defs>
<style>
  .lbl { font-family: sans-serif; font-size: 12px; fill: #555; }
  .atom { font-family: sans-serif; font-size: 14px; font-weight: 500; fill: #222; }
  .small { font-family: sans-serif; font-size: 11px; fill: #555; }
</style>

<!-- REACTANTS -->
<text class="lbl" x="120" y="28" text-anchor="middle">Reactants</text>
<text class="atom" x="48" y="128" text-anchor="middle" dominant-baseline="central">O</text>
<circle cx="38"  cy="116" r="2.5" fill="#444"/>
<circle cx="58"  cy="116" r="2.5" fill="#444"/>
<text class="small" x="62" y="112" text-anchor="start">(&#179;P)</text>
<text class="atom" x="100" y="128" text-anchor="middle" dominant-baseline="central">+</text>
<line x1="134" y1="118" x2="186" y2="118" stroke="#222" stroke-width="1.5"/>
<line x1="134" y1="126" x2="186" y2="126" stroke="#222" stroke-width="1.5"/>
<text class="atom" x="130" y="126" text-anchor="middle" dominant-baseline="central">C</text>
<text class="atom" x="190" y="126" text-anchor="middle" dominant-baseline="central">C</text>
<text class="small" x="104" y="100" text-anchor="middle">H</text>
<line x1="112" y1="104" x2="122" y2="116" stroke="#222" stroke-width="1.2"/>
<text class="small" x="104" y="152" text-anchor="middle">H</text>
<line x1="112" y1="148" x2="122" y2="132" stroke="#222" stroke-width="1.2"/>
<text class="small" x="216" y="100" text-anchor="middle">H</text>
<line x1="208" y1="104" x2="198" y2="116" stroke="#222" stroke-width="1.2"/>
<text class="small" x="216" y="152" text-anchor="middle">H</text>
<line x1="208" y1="148" x2="198" y2="132" stroke="#222" stroke-width="1.2"/>

<!-- ARROW -->
<line x1="240" y1="122" x2="300" y2="122" stroke="#888" stroke-width="1.5" marker-end="url(#arrow)"/>

<!-- TRANSITION STATE -->
<text class="lbl" x="430" y="28" text-anchor="middle">Transition state</text>
<text class="atom" x="342" y="72" text-anchor="middle" dominant-baseline="central">O</text>
<line x1="348" y1="82" x2="358" y2="108" stroke="#222" stroke-width="1.5" stroke-dasharray="3 2"/>
<circle cx="330" cy="62" r="2.5" fill="#444"/>
<line x1="366" y1="116" x2="424" y2="116" stroke="#888" stroke-width="1.2" stroke-dasharray="4 2"/>
<line x1="366" y1="124" x2="424" y2="124" stroke="#888" stroke-width="1.2" stroke-dasharray="4 2"/>
<text class="atom" x="360" y="124" text-anchor="middle" dominant-baseline="central">C</text>
<text class="atom" x="430" y="124" text-anchor="middle" dominant-baseline="central">C</text>
<circle cx="439" cy="113" r="2.5" fill="#C03020"/>
<text class="small" x="334" y="100" text-anchor="middle">H</text>
<line x1="342" y1="104" x2="353" y2="114" stroke="#222" stroke-width="1.2"/>
<text class="small" x="334" y="152" text-anchor="middle">H</text>
<line x1="342" y1="148" x2="353" y2="130" stroke="#222" stroke-width="1.2"/>
<text class="small" x="456" y="100" text-anchor="middle">H</text>
<line x1="448" y1="104" x2="438" y2="114" stroke="#222" stroke-width="1.2"/>
<text class="small" x="456" y="152" text-anchor="middle">H</text>
<line x1="448" y1="148" x2="438" y2="130" stroke="#222" stroke-width="1.2"/>
<path d="M310,46 L302,46 L302,158 L310,158" fill="none" stroke="#888" stroke-width="1.5"/>
<path d="M472,46 L480,46 L480,158 L472,158" fill="none" stroke="#888" stroke-width="1.5"/>
<text class="lbl" x="391" y="176" text-anchor="middle">&#8225;</text>
<path d="M370,114 Q360,85 350,82" fill="none" stroke="#1060A0" stroke-width="1.5" marker-end="url(#curl)"/>
<path d="M418,114 Q426,102 436,113" fill="none" stroke="#C03020" stroke-width="1.5" marker-end="url(#curl)"/>

<!-- ARROW -->
<line x1="496" y1="122" x2="546" y2="122" stroke="#888" stroke-width="1.5" marker-end="url(#arrow)"/>

<!-- PRODUCT -->
<text class="lbl" x="610" y="28" text-anchor="middle">Biradical</text>
<text class="atom" x="558" y="74" text-anchor="middle" dominant-baseline="central">O</text>
<line x1="562" y1="84" x2="572" y2="108" stroke="#222" stroke-width="1.5"/>
<circle cx="546" cy="64" r="2.5" fill="#444"/>
<line x1="578" y1="122" x2="628" y2="122" stroke="#222" stroke-width="1.5"/>
<text class="atom" x="572" y="126" text-anchor="middle" dominant-baseline="central">C</text>
<text class="atom" x="634" y="126" text-anchor="middle" dominant-baseline="central">C</text>
<circle cx="634" cy="114" r="2.5" fill="#C03020"/>
<text class="small" x="546" y="110" text-anchor="middle">H</text>
<line x1="554" y1="110" x2="566" y2="118" stroke="#222" stroke-width="1.2"/>
<text class="small" x="556" y="150" text-anchor="middle">H</text>
<line x1="560" y1="146" x2="568" y2="132" stroke="#222" stroke-width="1.2"/>
<text class="small" x="652" y="100" text-anchor="middle">H</text>
<line x1="646" y1="104" x2="638" y2="114" stroke="#222" stroke-width="1.2"/>
<text class="small" x="652" y="150" text-anchor="middle">H</text>
<line x1="646" y1="146" x2="638" y2="130" stroke="#222" stroke-width="1.2"/>

<!-- LEGEND -->
<line x1="40" y1="318" x2="80" y2="318" stroke="#888" stroke-width="1.5" stroke-dasharray="4 2"/>
<text class="small" x="88" y="322">Partial / forming bond</text>
<circle cx="55" cy="346" r="3" fill="#C03020"/>
<text class="small" x="88" y="350">Radical on carbon</text>
<circle cx="55" cy="370" r="3" fill="#444"/>
<text class="small" x="88" y="374">Radical on oxygen</text>
<path d="M260,336 Q270,326 280,333" fill="none" stroke="#1060A0" stroke-width="1.5" marker-end="url(#curl)"/>
<text class="small" x="290" y="342">π electrons &#8594; O&#8211;C bond</text>
<path d="M260,362 Q270,352 280,359" fill="none" stroke="#C03020" stroke-width="1.5" marker-end="url(#curl)"/>
<text class="small" x="290" y="368">π electrons &#8594; C2 radical</text>
</svg>
"""

display(SVG(reaction_scheme))

## 3. Starting geometry

The starting geometry places O at **3.5 Å from C1**, roughly perpendicular
to the ethylene plane. This choice reflects:

- **3.5 Å** is far enough that the electronic structure closely resembles
  separated reactants — a good approximation to the asymptote.
- **Perpendicular approach** to the π system is the expected trajectory
  for electrophilic addition of atomic oxygen.
- **Asymmetric attack** on C1 only — oxygen approaches one carbon,
  breaking the symmetry and allowing pyramidalization to develop.

The scan constrains the **O–C1 bond distance** (atoms 0 and 2, 0-indexed)
and relaxes all other degrees of freedom at each step.


In [ ]:
start_geometry = """
C    0.000000    0.000000    0.000000
C    1.335000    0.000000    0.000000
O   -0.517000    0.000000    1.932000
H   -0.585000    0.925000   -0.150000
H   -0.585000   -0.925000   -0.150000
H    1.920000    0.925000    0.050000
H    1.920000   -0.925000    0.050000
"""
print(start_geometry)

In [ ]:
import py3Dmol

xyz_block = f'7\nO(3P) + ethylene starting geometry  R(O-C1) = 3.5 Ang\n{start_geometry.strip()}'
view = py3Dmol.view(width=450, height=350)
view.addModel(xyz_block, 'xyz')
view.setStyle({'stick': {'colorscheme': 'grayCarbon', 'radius': 0.15},
               'sphere': {'scale': 0.25}})
view.setBackgroundColor('white')
view.zoomTo()
view.show()

## 4. Coarse CASSCF relaxed scan (10 points, 3.5 → 1.45 Å)

Active space: 6 electrons in 6 orbitals — the π/π* of ethylene and the two singly
occupied p orbitals on oxygen.

**Note:** NEVPT2 has no analytic gradient, so we use CASSCF for geometry relaxation
and apply NEVPT2 as single-point corrections in the next step.

**Note:** Do not use `PrintLevel` inside `%geom` when a `Scan` block is present —
ORCA's scan parser does not recognise it and will abort silently.


In [ ]:
coarse_scan_inp = f"""! CASSCF cc-pVDZ TightSCF Opt

{pal_block}%geom
  Scan
    B 0 2 = 3.5, 1.45, 10
  end
end

%casscf
  nel 6
  norb 6
  mult 3
  nroots 1
end

* xyz 0 3
{start_geometry}
*
"""

print('Starting coarse CASSCF scan ...')
print(coarse_scan_inp)
coarse_outfile = run_orca('scan_nevpt2', coarse_scan_inp, work_dir)
print('Done:', coarse_outfile)
print('Terminated normally:', terminated_normally(coarse_outfile))

## 5. NEVPT2 single points on coarse geometries

We use the `.gbw` file from each scan step as orbital guess via `MORead` —
CASSCF converges in very few iterations since the orbitals are already
optimised for that geometry.


In [ ]:
def run_nevpt2_singlepoints(work_dir, xyz_files, nprocs=1):
    work_dir = Path(work_dir)
    pal = f'%pal nprocs {nprocs} end\n\n' if nprocs > 1 else ''
    results = []
    for xyz_file in xyz_files:
        xyz_file = Path(xyz_file)
        lines = xyz_file.read_text().splitlines()
        geom = '\n'.join(lines[2:])
        gbw_file = xyz_file.with_suffix('.gbw').resolve()
        tag = 'nevpt2_sp_' + xyz_file.stem

        inp = f"""! NEVPT2 cc-pVDZ TightSCF MORead

{pal}%moinp "{gbw_file}"

%casscf
  nel 6
  norb 6
  mult 3
  nroots 1
end

* xyz 0 3
{geom}
*
"""
        print(f'  Running {tag} ...')
        outfile = run_orca(tag, inp, work_dir)
        E = get_nevpt2_energy(outfile)
        print(f'    E = {E:.6f} Eh')
        results.append(E)
    return np.array(results)


coarse_xyzs = sorted(work_dir.glob('scan_nevpt2.0*.xyz'))
print(f'Found {len(coarse_xyzs)} coarse geometries')
print('Starting NEVPT2 single points on coarse geometries ...')
nevpt2_coarse = run_nevpt2_singlepoints(work_dir, coarse_xyzs, nprocs=NPROCS)
print('All done.')

## 6. Dense CASSCF relaxed scan (2.4 → 1.45 Å)

Resolves the barrier region more finely.
Uses the geometry at R=2.36 Å (coarse step 6) as starting point.

**Note:** Do not extend below 1.45 Å — CASSCF convergence becomes unreliable
on the triplet surface at very short O–C distances where the singlet product
surface is nearby.


In [ ]:
dense_start_geom = work_dir / 'scan_nevpt2.006.xyz'
dense_geom_lines = dense_start_geom.read_text().splitlines()
dense_geom = '\n'.join(dense_geom_lines[2:])

dense_scan_inp = f"""! CASSCF cc-pVDZ TightSCF Opt

{pal_block}%geom
  Scan
    B 0 2 = 2.4, 1.45, 15
  end
end

%casscf
  nel 6
  norb 6
  mult 3
  nroots 1
end

* xyz 0 3
{dense_geom}
*
"""

print('Starting dense CASSCF scan ...')
dense_outfile = run_orca('scan_dense', dense_scan_inp, work_dir)
print('Done:', dense_outfile)
print('Terminated normally:', terminated_normally(dense_outfile))

## 7. NEVPT2 single points on dense geometries


In [ ]:
dense_xyzs = sorted(work_dir.glob('scan_dense.0*.xyz'))
print(f'Found {len(dense_xyzs)} dense geometries')
print('Starting NEVPT2 single points on dense geometries ...')
nevpt2_dense = run_nevpt2_singlepoints(work_dir, dense_xyzs, nprocs=NPROCS)
print('All done.')

## 8. B3LYP single points on all CASSCF geometries

Same geometries, same basis set — only the electronic structure method differs.
This isolates the effect of the method from any geometry differences.

**Note:** `STABPerform true` checks wavefunction stability and attempts reconvergence.
At several geometries in the approach region the triplet B3LYP solution is intrinsically
unstable — ORCA aborts those jobs after 5 attempts. These points appear as NaN in the
energy array and are skipped in the plot.


In [ ]:
def run_b3lyp_singlepoints(work_dir, xyz_files, nprocs=1):
    work_dir = Path(work_dir)
    pal = f'%pal nprocs {nprocs} end\n\n' if nprocs > 1 else ''
    results = []
    for xyz_file in xyz_files:
        xyz_file = Path(xyz_file)
        lines = xyz_file.read_text().splitlines()
        geom = '\n'.join(lines[2:])
        tag = 'b3lyp_sp_' + xyz_file.stem

        inp = f"""! B3LYP cc-pVDZ TightSCF SlowConv

{pal}%scf
  MaxIter 500
  STABPerform true
end

* xyz 0 3
{geom}
*
"""
        print(f'  Running {tag} ...')
        outfile = run_orca(tag, inp, work_dir)
        E = get_energy(outfile)
        print(f'    E = {E:.6f} Eh')
        results.append(E)
    return np.array(results)


all_xyzs = sorted(coarse_xyzs + dense_xyzs, key=get_oc_distance)
print(f'Total geometries: {len(all_xyzs)}')
print('Starting B3LYP single points ...')
b3lyp_energies = run_b3lyp_singlepoints(work_dir, all_xyzs, nprocs=NPROCS)
print('All done.')

## 9. Collect R values and merge datasets


In [ ]:
R_coarse = np.array([get_oc_distance(f) for f in coarse_xyzs])
R_dense  = np.array([get_oc_distance(f) for f in dense_xyzs])
R_all    = np.concatenate([R_coarse, R_dense])

sort_idx       = np.argsort(R_all)
R_all          = R_all[sort_idx]
nevpt2_all     = np.concatenate([nevpt2_coarse, nevpt2_dense])[sort_idx]
b3lyp_energies = b3lyp_energies[sort_idx]

coarse_dat  = np.loadtxt(work_dir / 'scan_nevpt2.relaxscanact.dat')
dense_dat   = np.loadtxt(work_dir / 'scan_dense.relaxscanact.dat')
R_casscf    = np.concatenate([coarse_dat[:, 0], dense_dat[:, 0]])
E_casscf    = np.concatenate([coarse_dat[:, 1], dense_dat[:, 1]])
casscf_sort = np.argsort(R_casscf)
R_casscf    = R_casscf[casscf_sort]
E_casscf    = E_casscf[casscf_sort]

E_casscf_rel = (E_casscf        - E_casscf[-1])        * HARTREE_TO_KJMOL
E_nevpt2_rel = (nevpt2_all      - nevpt2_all[-1])       * HARTREE_TO_KJMOL
E_b3lyp_rel  = np.where(
    np.isfinite(b3lyp_energies),
    (b3lyp_energies - np.nanmax(b3lyp_energies)) * HARTREE_TO_KJMOL,
    np.nan
)

print(f'R range: {R_all.min():.2f} - {R_all.max():.2f} \u00c5  ({len(R_all)} points)')
print(f'B3LYP convergence failures: {np.isnan(b3lyp_energies).sum()}')

## 10. Final energy profile plot


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(R_casscf, E_casscf_rel, 'o--', color='steelblue',   label='CASSCF(6,6)')
ax.plot(R_all,    E_nevpt2_rel, 'o-',  color='darkorange',  label='CASSCF + NEVPT2')
ax.plot(R_all,    E_b3lyp_rel,  'o-',  color='forestgreen', label='B3LYP')

ax.axhline(0, color='gray', lw=0.8, ls=':')
ax.set_xlabel('R(O\u2013C1) [\u00c5]', fontsize=12)
ax.set_ylabel('Relative energy [kJ/mol]', fontsize=12)
ax.set_title('O(\u00b3P) attack on ethylene: B3LYP vs CASSCF vs NEVPT2', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig(work_dir / 'final_profile.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to', work_dir / 'final_profile.png')

## 11. Build animation trajectory


In [ ]:
traj_path = work_dir / 'full_path.xyz'
all_xyz_sorted = build_xyz_trajectory(
    list(work_dir.glob('scan_nevpt2.0*.xyz')) +
    list(work_dir.glob('scan_dense.0*.xyz')),
    traj_path,
    key=get_oc_distance,
    reverse=True
)

R_frames = np.array([get_oc_distance(f) for f in all_xyz_sorted])
nevpt2_frames = np.array([
    get_nevpt2_energy(work_dir / f'nevpt2_sp_{Path(f).stem}.out')
    for f in all_xyz_sorted
])
E_frames = (nevpt2_frames - nevpt2_frames[-1]) * HARTREE_TO_KJMOL

print(f'Trajectory: {len(R_frames)} frames, R = {R_frames.max():.2f} \u2192 {R_frames.min():.2f} \u00c5')

## 12. Interactive trajectory viewer

Step through frames using the nglview player controls (hover over the molecule).
The red dot on the energy curve tracks the current geometry.

Things to look for:
- Pyramidalization of C1 as O approaches
- C=C bond lengthening through the barrier region
- Geometry of the triplet biradical intermediate at short R


In [ ]:
import nglview
%matplotlib widget

traj = load_xyz_as_traj(str(traj_path), silent=True)
view = nglview.show_asetraj(traj)
view._set_size('400px', '350px')

plt.ioff()
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(R_frames, E_frames, 'o-', color='darkorange', label='NEVPT2')
marker, = ax.plot([R_frames[0]], [E_frames[0]], 'ro', ms=10)
ax.set_xlabel('R(O\u2013C1) [\u00c5]')
ax.set_ylabel('Relative energy [kJ/mol]')
ax.grid(True, alpha=0.4)
ax.legend()
plt.tight_layout()
fig.canvas.layout = widgets.Layout(width='400px', height='350px')

def on_frame_change(change):
    i = change['new']
    marker.set_data([R_frames[i]], [E_frames[i]])
    fig.canvas.draw_idle()

view.observe(on_frame_change, names=['frame'])

panel = widgets.HBox([view, fig.canvas])
display(panel)
plt.ion()

## 13. Questions for Students

1. **Why does B3LYP predict no barrier?**  
   Think about the electronic structure of O(³P) and what happens to the spin as it approaches the π system.

2. **Why does B3LYP fail to converge at several geometries in the approach region?**  
   What does a negative stability eigenvalue mean physically?

3. **Why does CASSCF overestimate the barrier?**  
   What kind of electron correlation is CASSCF missing? What does NEVPT2 add?

4. **What is the active space (6,6) capturing here?**  
   Which orbitals would you expect to be in the active space at the transition state?

5. **How does the geometry change along the reaction path?**  
   Use the animation to identify when pyramidalization of C1 begins.
   What does this tell us about the timing of bond formation?

6. **How does spin density change along the reaction path?**  
   Where does the unpaired spin end up in the product?
   What kind of intermediate is formed?
